# 🚗🏍️ RAG Chatbot for Bike & Car Prices/Mileage — Practice Notebook

This notebook walks through building a **Retrieval-Augmented Generation (RAG)**
chatbot step-by-step, so you understand every piece before using the
production `.py` files in `../backend/`.

**Stack:**
- **Data**: PDF spec-sheets (`../data/bikes_data.pdf`, `../data/cars_data.pdf`)
- **Vector DB**: [ChromaDB](https://www.trychroma.com/) — runs 100% locally, no account needed
- **Embeddings**: local `sentence-transformers` model (`all-MiniLM-L6-v2`) — free, no API key
- **LLM**: [OpenRouter](https://openrouter.ai/) — one API key, access to many models (some free)

**Pipeline:**
```
PDF files -> extract text -> chunk -> embed -> store in ChromaDB
                                                        |
User question -> embed -> similarity search  <---------+
                    |
            retrieved chunks + question -> OpenRouter LLM -> grounded answer
```


## 0. Setup — install & import dependencies

In [1]:
# Run once. In Colab/fresh envs uncomment the pip install line.
# !pip install chromadb sentence-transformers pdfplumber openai python-dotenv reportlab -q

import os
import re
from pathlib import Path

import pdfplumber
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI

DATA_DIR = Path("../data")
CHROMA_DIR = Path("./chroma_store_notebook")  # separate store just for this notebook
CHROMA_DIR.mkdir(exist_ok=True)


## 1. Set your OpenRouter API key

Get a free key at **https://openrouter.ai/keys**.

Never hardcode your key in a notebook you'll share — for practice we read it from
an environment variable or prompt for it securely.


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

print("Current working dir:", os.getcwd())
print(".env in cwd:", os.path.exists(".env"))
print(".env in parent:", os.path.exists("../.env"))

load_dotenv(dotenv_path="../.env")  # adjust path based on what printed above
print("After load_dotenv, key found:", os.getenv("OPENROUTER_API_KEY"))

Current working dir: c:\Users\goura\anaconda3\GEN_AI\project\notebook
.env in cwd: False
.env in parent: True
After load_dotenv, key found: None


In [3]:
from pathlib import Path

project_dir = Path("../")
print("Files in project folder:")
for f in project_dir.iterdir():
    print(" -", f.name)

Files in project folder:
 - .env
 - backend
 - data
 - frontend
 - notebook
 - README.md
 - {data,notebook,backend,frontend}


In [5]:
from pathlib import Path

env_path = Path("../.env")
print("Exists:", env_path.exists())
print("Raw content:\n", env_path.read_text())

Exists: True
Raw content:
 


In [6]:
from dotenv import load_dotenv
from pathlib import Path
import os, getpass

# .env lives one folder up from this notebook (in project/), adjust if needed
env_path = Path("../.env")
load_dotenv(dotenv_path=env_path)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass.getpass("Enter your OpenRouter API key: ")
assert OPENROUTER_API_KEY, "You need an OpenRouter API key to continue."
print("Key loaded ✔")

Key loaded ✔


## 2. Look at the raw data

We generated two sample PDFs: one for bikes, one for cars. Each contains a
"Quick Overview" table plus one detailed text block per vehicle (brand, model,
variant, fuel, engine, mileage, ex-showroom price, on-road price).

> 📌 In a real project, swap these demo PDFs for verified, up-to-date data
> (dealer sheets, manufacturer spec PDFs, scraped & cleaned listings, etc.)


In [7]:
list(DATA_DIR.glob("*.pdf"))


[WindowsPath('../data/bikes_data.pdf'), WindowsPath('../data/cars_data.pdf')]

In [8]:
def extract_text_from_pdf(pdf_path):
    text_parts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text_parts.append(page.extract_text() or "")
    return "\n".join(text_parts)

bike_text = extract_text_from_pdf(DATA_DIR / "bikes_data.pdf")
car_text = extract_text_from_pdf(DATA_DIR / "cars_data.pdf")

print(bike_text[:800])


Two-Wheeler Price & Mileage Guide
Sample dataset of popular bikes/scooters/EVs in India with ex-showroom price, on-road price
(Bangalore) and mileage. Demo data for practising RAG pipelines — replace with live/verified data
before using in production.
Quick Overview
Brand Model Fuel Mileage On-Road Price
Hero Splendor Plus Petrol 65 km/l Rs. 93,200
Hero HF Deluxe Petrol 70 km/l Rs. 76,800
Honda Activa 6G Petrol 50 km/l Rs. 94,500
Honda Shine 100 Petrol 66 km/l Rs. 80,300
Honda SP 125 Petrol 58 km/l Rs. 109,500
Bajaj Platina 100 Petrol 70 km/l Rs. 79,600
Bajaj Pulsar 150 Petrol 45 km/l Rs. 128,500
Bajaj Pulsar NS200 Petrol 35 km/l Rs. 178,900
TVS Jupiter 110 Petrol 50 km/l Rs. 92,800
TVS Apache RTR 160 4V Petrol 45 km/l Rs. 146,200
TVS Apache RTR 310 Petrol 32 km/l Rs. 291,500
Yamaha FZ-S F


In [ ]:
print(car_text[:800])

## 3. Chunking

LLMs and vector search work best on small, focused pieces of text ("chunks"),
not entire documents. Our PDFs are built **one vehicle per paragraph block**,
so we chunk on blank-line boundaries — this keeps every chunk a complete,
unambiguous vehicle record (no mid-sentence cuts, no mixed vehicles).


In [ ]:
def chunk_text(text, chunk_size=600, overlap=80):
    blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if b.strip()]
    chunks, buffer = [], ""
    for block in blocks:
        if len(buffer) + len(block) + 1 <= chunk_size:
            buffer = f"{buffer}\n{block}".strip()
        else:
            if buffer:
                chunks.append(buffer)
            if len(block) <= chunk_size:
                buffer = block
            else:
                start = 0
                while start < len(block):
                    end = start + chunk_size
                    chunks.append(block[start:end])
                    start = end - overlap
                buffer = ""
    if buffer:
        chunks.append(buffer)
    return chunks

bike_chunks = chunk_text(bike_text)
car_chunks = chunk_text(car_text)
print(f"Bike chunks: {len(bike_chunks)}, Car chunks: {len(car_chunks)}")
print("\n--- Example chunk ---\n")
print(bike_chunks[5])


## 4. Embeddings + ChromaDB

We use a **local** embedding model (`all-MiniLM-L6-v2` via `sentence-transformers`)
so there's no API key or account needed for the vector DB side. ChromaDB stores
everything on disk in `CHROMA_DIR` (a `PersistentClient`) — again, no cloud
account required.


In [ ]:
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path=str(CHROMA_DIR))

# Fresh collection each run of this notebook
try:
    client.delete_collection("vehicle_prices_demo")
except Exception:
    pass

collection = client.create_collection(
    name="vehicle_prices_demo",
    embedding_function=embed_fn,
    metadata={"hnsw:space": "cosine"},
)
print("Collection created ✔")


In [ ]:
ids, docs, metas = [], [], []

for i, c in enumerate(bike_chunks):
    ids.append(f"bike_{i}")
    docs.append(c)
    metas.append({"source": "bikes_data.pdf", "category": "bike"})

for i, c in enumerate(car_chunks):
    ids.append(f"car_{i}")
    docs.append(c)
    metas.append({"source": "cars_data.pdf", "category": "car"})

collection.add(ids=ids, documents=docs, metadatas=metas)
print(f"Ingested {len(ids)} chunks into ChromaDB ✔")


## 5. Try retrieval on its own (before adding the LLM)

In [ ]:
def retrieve(query, top_k=4):
    results = collection.query(query_texts=[query], n_results=top_k)
    return list(zip(results["documents"][0], results["metadatas"][0], results["distances"][0]))

for doc, meta, dist in retrieve("on road price of Honda Activa"):
    print(f"[{meta['category']} | dist={dist:.3f}] {doc[:120]}...")
    print("---")


## 6. Add the LLM via OpenRouter

OpenRouter exposes an **OpenAI-compatible API**, so we just point the
standard `openai` Python client at OpenRouter's base URL.


In [ ]:
llm = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

SYSTEM_PROMPT = '''You are a helpful assistant that answers questions about bike and car
prices and mileage, using ONLY the CONTEXT provided.
Always state On-Road Price, Ex-Showroom Price and Mileage clearly when available.
If multiple vehicles match, list each separately.
If the answer isn't in the context, say so honestly instead of guessing.'''

def ask_llm(query, chunks):
    context = "\n\n---\n\n".join(c[0] for c in chunks)
    user_prompt = f"CONTEXT:\n{context}\n\nQUESTION:\n{query}"
    response = llm.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.2,
    )
    return response.choices[0].message.content


## 7. Put it all together — full RAG query function

In [ ]:
def rag_ask(query, top_k=4):
    chunks = retrieve(query, top_k=top_k)
    answer = ask_llm(query, chunks)
    return answer, chunks

answer, used_chunks = rag_ask("What is the on-road price and mileage of Honda Activa 6G?")
print(answer)


In [ ]:
# Try a few more questions
for q in [
    "Compare the on-road price of Tata Nexon and Hyundai Creta",
    "Which is the cheapest electric bike?",
    "What is the mileage of Royal Enfield Classic 350?",
]:
    print("Q:", q)
    ans, _ = rag_ask(q)
    print("A:", ans)
    print("=" * 80)


## 8. Next steps

- The equivalent, cleaned-up **production code** lives in `../backend/`:
  - `config.py` — env-var driven configuration
  - `ingest.py` — reusable ingestion pipeline (run once per data update)
  - `rag_pipeline.py` — a `RAGPipeline` class wrapping retrieval + generation
- The **Streamlit UI** is in `../frontend/app.py` — run with:
  ```bash
  streamlit run ../frontend/app.py
  ```
- Swap in your own PDFs (dealer price lists, manufacturer spec sheets) inside
  `../data/`, then re-run `python ../backend/ingest.py` to rebuild the vector store.
- Try swapping `OPENROUTER_MODEL` for a different model on
  [openrouter.ai/models](https://openrouter.ai/models) to compare quality/cost.
